In [ ]:
# ==============================================================================
# CELL 00 — TPU VM BOOTSTRAP (must run BEFORE any `import tensorflow`)
#
# Kaggle TPU v5e-8 sessions run on TPU VMs whose preinstalled TF pip build
# cannot see the TPU until the version-matched PJRT plugin (tensorflow-tpu
# + libtpu) is installed. This cell:
#   1. Probes for TPU hardware via JAX (ground truth on PJRT-based TPU VMs;
#      /dev/accel* does NOT exist on v5e PJRT runtimes)
#   2. Installs tensorflow-tpu pinned to the preinstalled TF version
#   3. Fails fast with actionable guidance if no TPU hardware is present
# ==============================================================================

import os, sys, glob, subprocess
from importlib import metadata as _metadata

print(f"Python: {sys.version.split()[0]}")

# ── 1. Hardware probe ──
# On Kaggle TPU v5e VMs, the PJRT runtime does NOT expose /dev/accel*
# character devices. The reliable ground truth is JAX's device list.
accel_devs = sorted(glob.glob("/dev/accel*"))
print(f"/dev/accel* devices: {accel_devs if accel_devs else 'NONE (expected on PJRT v5e)'}")
print(f"TPU_NAME env: {os.environ.get('TPU_NAME')!r}")

# JAX probe — the real check for TPU presence on PJRT VMs
_tpu_visible = False
try:
    import jax
    _jax_devs = jax.devices()
    _tpu_visible = any(d.platform == 'tpu' for d in _jax_devs)
    print(f"jax.devices(): {_jax_devs}")
    print(f"TPU visible via JAX: {_tpu_visible}")
except Exception as e:
    print(f"jax probe skipped/failed: {type(e).__name__}: {e}")

try:
    tf_ver = _metadata.version("tensorflow")
except Exception:
    tf_ver = None
print(f"preinstalled tensorflow dist version: {tf_ver}")
try:
    _metadata.version("tensorflow-tpu")
    _have_plugin = True
except Exception:
    _have_plugin = False
print(f"tensorflow-tpu plugin present: {_have_plugin}")

# Fail-fast: neither /dev/accel* NOR JAX can see a TPU
if not accel_devs and not _tpu_visible:
    raise RuntimeError(
        "\n============================================================\n"
        "NO TPU HARDWARE VISIBLE\n"
        "Neither /dev/accel* nor JAX can detect a TPU on this VM.\n"
        "This run did not land on a TPU VM host. Checklist:\n"
        "- TPU quota remaining (20h/week free tier)\n"
        "- No other batch TPU session active (limit: 1 per user)\n"
        "- kernel-metadata.json has machine_shape=TpuV5E8 and the push\n"
        "  used --accelerator TpuV5E8 (enable_tpu alone => CPU image)\n"
        "Then re-push to re-enter the batch TPU queue.\n"
        "============================================================"
    )

# ── 2. Install version-matched PJRT plugin (TPU-enabled TF build) ──
# The tensorflow-tpu wheel provides the TPU-enabled tensorflow module files;
# installing it BEFORE the first `import tensorflow` means the import in the
# next cell picks up TPU support with no kernel restart on a fresh worker.
# Pin to the preinstalled TF version so plugin and framework stay in sync.
if not _have_plugin:
    pin = tf_ver if tf_ver else "2.20.0"
    print(f"Installing tensorflow-tpu=={pin} (provides libtpu + TPU kernels)...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q",
         "--progress-bar", "off", f"tensorflow-tpu=={pin}"]
    )
    print(f"tensorflow-tpu installed: {_metadata.version('tensorflow-tpu')}")
else:
    print("tensorflow-tpu already present — skipping install.")


In [ ]:
# ==============================================================================
# ⚡ ALPHA 450M — SEQUENTIAL TPU v5e-8 OPTIMIZER BENCHMARK
#
# Evaluates 3 optimizer configurations sequentially on ~50M tokens each:
# 1. hybrid_muon_adamw : Muon for 2D hidden weights, AdamW for 1D/embeddings
# 2. adamw             : Built-in Keras AdamW (reliability baseline)
# 3. muon              : Pure Muon for 2D weights, momentum SGD for 1D
#
# TPU Optimizations:
# - mixed_bfloat16 global policy with strict dtype matching
# - Safe per-replica batch size (4 seqs/core = 65,536 tok/step) preventing HBM OOM
# - steps_per_execution=10 (XLA fusion for benchmark)
# - Precomputed static RoPE (pre-cast bfloat16)
# - Tile-reshape GQA KV expansion (4:1)
# - distribute_datasets_from_function with drop_remainder=True
# - Strict fail-hard TPU initialization (no silent CPU fallback)
# ==============================================================================

import os, sys, time, math, json, gc
import numpy as np
import tensorflow as tf

print(f"TensorFlow: {tf.__version__} | Python: {sys.version.split()[0]}")
tf.keras.mixed_precision.set_global_policy("mixed_bfloat16")

# ── TPU diagnostics (helps debug accelerator misconfiguration) ──
print(f"TPU_NAME env: {os.environ.get('TPU_NAME')!r}")
print(f"COLAB_TPU_ADDR env: {os.environ.get('COLAB_TPU_ADDR')!r}")
print(f"Physical devices: {tf.config.list_physical_devices()}")
try:
    print(f"Logical TPU (pre-init): {tf.config.list_logical_devices('TPU')}")
except Exception as _e:
    print(f"Logical TPU (pre-init) query failed: {_e}")

# ── TPU initialization: FAIL HARD if unavailable ──
# Proven Kaggle pattern: TPUClusterResolver() with no arguments auto-detects
# the TPU address from the environment, then explicit connect + initialize.
# This is the same pattern used in the training notebook and matches Kaggle
# docs for both v3-8 and v5e-8.
# ── TPU initialization: FAIL HARD if unavailable ──
# Proven Kaggle pattern: TPUClusterResolver() with no arguments auto-detects
# the TPU address from the environment, then explicit connect + initialize.
# This is the same pattern used in the training notebook and matches Kaggle
# docs for both v3-8 and v5e-8.
# Fallbacks (this TPU VM sets no TPU_NAME, so no-arg construction can fail
# before init is attempted):
#  2. TPUClusterResolver(tpu="local") + initialize — directly-attached TPU
#     (works now that CELL 00 installed the libtpu plugin)
#  3. Default local init — assumes this host IS the TPU worker
tpu = None
_how = None
_attempt_errors = []
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    print(f"TPU master: {tpu.master()}")

    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    _how = "auto-detect"
except Exception as e:
    _attempt_errors.append(f"[auto-detect] {type(e).__name__}: {e}")
    try:
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver(tpu="local")
        tf.tpu.experimental.initialize_tpu_system(tpu)
        _how = "tpu='local'"
    except Exception as e2:
        _attempt_errors.append(f"[tpu='local'] {type(e2).__name__}: {e2}")
        try:
            tf.tpu.experimental.initialize_tpu_system()
            tpu = None
            _how = "local-default"
        except Exception as e3:
            _attempt_errors.append(f"[local-default] {type(e3).__name__}: {e3}")

if _how is None:
    raise RuntimeError(
        "\n"
        "============================================================\n"
        "TPU INITIALIZATION FAILED (all strategies exhausted)\n"
        "This benchmark is TPU-only; CPU fallback is disabled.\n"
        + "".join(f"  {m}\n" for m in _attempt_errors) +
        "On Kaggle: Settings -> Accelerator -> select TPU (v3-8 / v5e),\n"
        "Internet ON, then Save & Run All. If pushed via API:\n"
        "kernel-metadata.json must set machine_shape to TpuV5E8\n"
        "(or pass --accelerator TpuV5E8); enable_tpu alone schedules\n"
        "the run on the standard CPU image.\n"
        "============================================================"
    )

strategy = tf.distribute.TPUStrategy(tpu) if tpu is not None else tf.distribute.TPUStrategy()
print(f"TPU connected via {_how}")


print(f"TPU connected")


# Verify actual TPU devices
tpu_devices = tf.config.list_logical_devices("TPU")

if not tpu_devices:
    raise RuntimeError(
        "TPUStrategy was created, but no TPU devices are visible."
    )

print(f"✅ TPU active")
print(f"   Replicas: {strategy.num_replicas_in_sync}")
print(f"   TPU devices: {len(tpu_devices)}")

if strategy.num_replicas_in_sync != 8:
    raise RuntimeError(
        f"Expected TPU v5e-8 / v3-8 (8 replicas), "
        f"but detected {strategy.num_replicas_in_sync} replicas."
    )

# ── Architecture Constants ──
SEQUENCE_LENGTH    = 2048
VOCAB_SIZE         = 48000
HIDDEN_SIZE        = 1280
INTERMEDIATE_SIZE  = 3456
NUM_LAYERS         = 22
NUM_HEADS          = 20
NUM_KV_HEADS       = 5
HEAD_DIM           = HIDDEN_SIZE // NUM_HEADS   # 64
GQA_REP            = NUM_HEADS // NUM_KV_HEADS  # 4

# ── Benchmark Constants (Tuned for 16GB HBM Safety) ──
PER_REPLICA_BATCH  = 4   # 4 seqs/core prevents TPU v5e HBM OOM (0.62GB attention matrix)
GLOBAL_BATCH_SIZE  = PER_REPLICA_BATCH * strategy.num_replicas_in_sync  # 32 seqs on 8 cores
TOKENS_PER_STEP    = GLOBAL_BATCH_SIZE * SEQUENCE_LENGTH               # 65,536 tokens/step
STEPS_PER_EXEC     = 10  # XLA-fused steps per host round-trip
WARMUP_STEPS       = 10  # Triggers XLA compilation (not timed)
BENCH_STEPS        = 760 # 760 steps x 65,536 = ~49.8M tokens (~50M tokens)

print(f"Global Batch: {GLOBAL_BATCH_SIZE} seqs | {TOKENS_PER_STEP:,} tokens/step")
print(f"Benchmark: {BENCH_STEPS} steps = {BENCH_STEPS * TOKENS_PER_STEP:,} tokens per optimizer")


In [ ]:
# ==============================================================================
# CELL 02 — MODEL ARCHITECTURE (Static Shapes, Pre-Cast RoPE, Tile-Reshape GQA)
# ==============================================================================

from tensorflow.keras import layers, models


class KerasRMSNorm(layers.Layer):
    """Root Mean Square Layer Normalization with learnable scaling (float32-stable)."""
    def __init__(self, dim, eps=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.eps, self.dim = eps, dim

    def build(self, input_shape):
        self.weight = self.add_weight(shape=(self.dim,), name="weight", initializer="ones")

    def call(self, x):
        # float32 variance path: bfloat16 variance overflows/underflows on TPU
        variance = tf.reduce_mean(tf.square(tf.cast(x, tf.float32)), axis=-1, keepdims=True)
        normed = tf.cast(x, tf.float32) * tf.math.rsqrt(variance + tf.cast(self.eps, tf.float32))
        return tf.cast(normed * tf.cast(self.weight, tf.float32), x.dtype)

    def get_config(self):
        config = super().get_config()
        config.update({"dim": self.dim, "eps": self.eps})
        return config


class PrecomputedRoPE(layers.Layer):
    """Static RoPE tables pre-cast to bfloat16 at init — zero per-call overhead."""
    def __init__(self, head_dim=HEAD_DIM, max_ctx=SEQUENCE_LENGTH, theta=100000.0, **kwargs):
        super().__init__(**kwargs)
        self.head_dim = head_dim
        indices = tf.range(0, head_dim, 2, dtype=tf.float32)
        inv_freq = 1.0 / (theta ** (indices / head_dim))
        t = tf.range(max_ctx, dtype=tf.float32)
        freqs = tf.einsum("i,j->ij", t, inv_freq)
        cos_tab = tf.concat([tf.cos(freqs), tf.cos(freqs)], axis=-1)
        sin_tab = tf.concat([tf.sin(freqs), tf.sin(freqs)], axis=-1)
        # Pre-cast to bfloat16 + add [1, 1, T, D] broadcast dims
        self.cos = tf.cast(cos_tab[tf.newaxis, tf.newaxis, :, :], tf.bfloat16)
        self.sin = tf.cast(sin_tab[tf.newaxis, tf.newaxis, :, :], tf.bfloat16)

    def call(self, x):
        d = self.head_dim
        x1, x2 = x[..., :d // 2], x[..., d // 2:]
        cos = tf.cast(self.cos, x.dtype)
        sin = tf.cast(self.sin, x.dtype)
        return (x * cos) + (tf.concat([-x2, x1], axis=-1) * sin)


class OptimizedGQA(layers.Layer):
    """GQA with QK-Norm, pre-computed RoPE, static causal mask, tile-reshape KV expansion."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.q_proj = layers.Dense(NUM_HEADS * HEAD_DIM, use_bias=False, name="q_proj")
        self.k_proj = layers.Dense(NUM_KV_HEADS * HEAD_DIM, use_bias=False, name="k_proj")
        self.v_proj = layers.Dense(NUM_KV_HEADS * HEAD_DIM, use_bias=False, name="v_proj")
        self.o_proj = layers.Dense(HIDDEN_SIZE, use_bias=False, name="o_proj")
        self.q_norm = KerasRMSNorm(HEAD_DIM)
        self.k_norm = KerasRMSNorm(HEAD_DIM)
        self.rope = PrecomputedRoPE()
        # Float32 band_part then cast to bool with [1, 1, T, T] shape for clean broadcasting
        float_mask = tf.linalg.band_part(tf.ones((SEQUENCE_LENGTH, SEQUENCE_LENGTH), dtype=tf.float32), -1, 0)
        self.causal_mask = tf.cast(float_mask, tf.bool)[tf.newaxis, tf.newaxis, :, :]
        self.scale = tf.cast(HEAD_DIM ** -0.5, tf.bfloat16)

    def call(self, x):
        B = tf.shape(x)[0]
        T = SEQUENCE_LENGTH  # Static compile-time constant
        q = tf.reshape(self.q_proj(x), (B, T, NUM_HEADS, HEAD_DIM))
        k = tf.reshape(self.k_proj(x), (B, T, NUM_KV_HEADS, HEAD_DIM))
        v = tf.reshape(self.v_proj(x), (B, T, NUM_KV_HEADS, HEAD_DIM))
        # QK-Norm then RoPE, transpose to [B, H, T, D]
        q = self.rope(tf.transpose(self.q_norm(q), [0, 2, 1, 3]))
        k = self.rope(tf.transpose(self.k_norm(k), [0, 2, 1, 3]))
        v = tf.transpose(v, [0, 2, 1, 3])
        # GQA KV expansion via tile+reshape (XLA-friendly, avoids tf.repeat intermediates)
        k = tf.reshape(
            tf.tile(k[:, :, tf.newaxis, :, :], [1, 1, GQA_REP, 1, 1]),
            (B, NUM_HEADS, T, HEAD_DIM)
        )
        v = tf.reshape(
            tf.tile(v[:, :, tf.newaxis, :, :], [1, 1, GQA_REP, 1, 1]),
            (B, NUM_HEADS, T, HEAD_DIM)
        )
        # Scaled dot-product attention with static causal mask (-1e4 bfloat16-safe fill)
        scores = tf.matmul(q, k, transpose_b=True) * tf.cast(self.scale, q.dtype)
        scores = tf.where(self.causal_mask, scores, tf.cast(-1e4, scores.dtype))
        attn = tf.nn.softmax(scores, axis=-1)
        out = tf.matmul(attn, v)
        out = tf.reshape(tf.transpose(out, [0, 2, 1, 3]), (B, T, HIDDEN_SIZE))
        return self.o_proj(out)


class TransformerBlock(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.attn_norm = KerasRMSNorm(HIDDEN_SIZE)
        self.attn = OptimizedGQA()
        self.ffn_norm = KerasRMSNorm(HIDDEN_SIZE)
        self.gate = layers.Dense(INTERMEDIATE_SIZE, use_bias=False, name="gate_proj")
        self.up = layers.Dense(INTERMEDIATE_SIZE, use_bias=False, name="up_proj")
        self.down = layers.Dense(HIDDEN_SIZE, use_bias=False, name="down_proj")

    def call(self, x):
        x = x + self.attn(self.attn_norm(x))
        h = self.ffn_norm(x)
        x = x + self.down(tf.nn.silu(self.gate(h)) * self.up(h))
        return x


In [ ]:
# ==============================================================================
# CELL 03 — OPTIMIZER ENGINE (Newton-Schulz + HybridMuonAdamW + PureMuon + Factory)
# ==============================================================================

def zeropower_via_newtonschulz5(G, steps=5, eps=1e-7):
    """Compute G @ (G^T G)^{-1/2} via 5th-order Newton-Schulz iteration on TPU MXUs.
    Uses explicit bfloat16 constants to prevent TensorFlow dtype mismatch errors."""
    a = tf.cast(3.4445, tf.bfloat16)
    b = tf.cast(-4.7750, tf.bfloat16)
    c = tf.cast(2.0315, tf.bfloat16)
    eps_val = tf.cast(eps, tf.bfloat16)

    X = tf.cast(G, tf.bfloat16)
    X = X / (tf.norm(X) + eps_val)
    rows, cols = int(X.shape[0]), int(X.shape[1])  # Guaranteed Python ints
    transposed = False
    if rows > cols:
        X = tf.transpose(X)
        transposed = True
    for _ in range(steps):
        A = tf.matmul(X, X, transpose_b=True)
        B = b * A + c * tf.matmul(A, A)
        X = a * X + tf.matmul(B, X)
    if transposed:
        X = tf.transpose(X)
    return tf.cast(X, G.dtype)


class HybridMuonAdamW(tf.keras.optimizers.Optimizer):
    """
    Hybrid optimizer matching PyTorch Muon semantics:
    - 2D hidden weights: Nesterov momentum -> Newton-Schulz orthogonalization
    - 1D/embeddings/norms: Full AdamW with bias-corrected first & second moments
    Weight decay uses the correct LR for each parameter group.
    """
    def __init__(self, lr_muon=0.02, lr_adam=5e-4, muon_momentum=0.95,
                 beta1=0.9, beta2=0.95, weight_decay=0.01, ns_steps=5,
                 clipnorm=1.0, **kwargs):
        super().__init__(learning_rate=lr_muon, clipnorm=clipnorm, **kwargs)
        self.lr_muon = lr_muon
        self.lr_adam = lr_adam
        self.muon_momentum = muon_momentum
        self.beta1 = beta1
        self.beta2 = beta2
        self.weight_decay = weight_decay
        self.ns_steps = ns_steps

    def build(self, var_list):
        super().build(var_list)
        # Type-selective allocation: 2D matrices get muon_buf only; 1D/embeddings get adam_m/v only.
        # Saves ~3GB HBM across 444M parameters on 16GB TPU v5e cores.
        self.muon_bufs = [
            self.add_variable_from_reference(v, name="muon_buf") if self._is_muon_param(v) else None
            for v in var_list
        ]
        self.adam_m = [
            self.add_variable_from_reference(v, name="adam_m") if not self._is_muon_param(v) else None
            for v in var_list
        ]
        self.adam_v = [
            self.add_variable_from_reference(v, name="adam_v") if not self._is_muon_param(v) else None
            for v in var_list
        ]

    def _is_muon_param(self, var):
        """2D hidden weight matrices -> Muon; everything else -> AdamW."""
        return len(var.shape) == 2 and "tok" not in var.name

    def update_step(self, gradient, variable, learning_rate):
        # Embedding grads arrive as IndexedSlices on TPU — densify first
        if isinstance(gradient, tf.IndexedSlices):
            gradient = tf.convert_to_tensor(gradient)
        idx = self._get_variable_index(variable)

        if self._is_muon_param(variable):
            # === MUON PATH (2D hidden weights) ===
            if self.weight_decay > 0:
                variable.assign(variable * tf.cast(1.0 - self.lr_muon * self.weight_decay, variable.dtype))
            buf = self.muon_bufs[idx]
            buf.assign(self.muon_momentum * buf + gradient)
            update = gradient + self.muon_momentum * buf
            scale = tf.sqrt(tf.constant(float(max(variable.shape)), dtype=tf.float32))
            ortho = zeropower_via_newtonschulz5(update, steps=self.ns_steps)
            step_update = tf.cast(self.lr_muon * scale, ortho.dtype) * ortho
            variable.assign_sub(tf.cast(step_update, variable.dtype))
        else:
            # === ADAMW PATH (1D, embeddings, norms) ===
            if self.weight_decay > 0:
                variable.assign(variable * tf.cast(1.0 - self.lr_adam * self.weight_decay, variable.dtype))
            t = tf.cast(self.iterations + 1, tf.float32)
            m = self.adam_m[idx]
            v = self.adam_v[idx]
            m.assign(self.beta1 * m + (1.0 - self.beta1) * gradient)
            v.assign(self.beta2 * v + (1.0 - self.beta2) * tf.square(gradient))
            m_hat = m / (1.0 - tf.pow(self.beta1, t))
            v_hat = v / (1.0 - tf.pow(self.beta2, t))
            step_update = self.lr_adam * m_hat / (tf.sqrt(v_hat) + 1e-8)
            variable.assign_sub(tf.cast(step_update, variable.dtype))

    def get_config(self):
        config = super().get_config()
        config.update({"lr_muon": self.lr_muon, "lr_adam": self.lr_adam,
                       "muon_momentum": self.muon_momentum, "beta1": self.beta1,
                       "beta2": self.beta2, "weight_decay": self.weight_decay,
                       "ns_steps": self.ns_steps})
        return config


class PureMuon(tf.keras.optimizers.Optimizer):
    """Pure Muon: Newton-Schulz for 2D hidden weights, Nesterov SGD for 1D/embeddings."""
    def __init__(self, lr_muon=0.02, lr_1d=5e-4, momentum=0.95,
                 weight_decay=0.01, ns_steps=5, clipnorm=1.0, **kwargs):
        super().__init__(learning_rate=lr_muon, clipnorm=clipnorm, **kwargs)
        self.lr_muon = lr_muon
        self.lr_1d = lr_1d
        self.momentum_val = momentum
        self.weight_decay = weight_decay
        self.ns_steps = ns_steps

    def build(self, var_list):
        super().build(var_list)
        self.bufs = [self.add_variable_from_reference(v, name="buf") for v in var_list]

    def update_step(self, gradient, variable, learning_rate):
        # Embedding grads arrive as IndexedSlices on TPU — densify first
        if isinstance(gradient, tf.IndexedSlices):
            gradient = tf.convert_to_tensor(gradient)
        idx = self._get_variable_index(variable)
        is_2d = len(variable.shape) == 2 and "tok" not in variable.name
        wd_lr = self.lr_muon if is_2d else self.lr_1d
        if self.weight_decay > 0:
            variable.assign(variable * tf.cast(1.0 - wd_lr * self.weight_decay, variable.dtype))
        buf = self.bufs[idx]
        buf.assign(self.momentum_val * buf + gradient)
        update = gradient + self.momentum_val * buf
        if is_2d:
            scale = tf.sqrt(tf.constant(float(max(variable.shape)), dtype=tf.float32))
            ortho = zeropower_via_newtonschulz5(update, steps=self.ns_steps)
            step_update = tf.cast(self.lr_muon * scale, ortho.dtype) * ortho
            variable.assign_sub(tf.cast(step_update, variable.dtype))
        else:
            variable.assign_sub(tf.cast(self.lr_1d * update, variable.dtype))

    def get_config(self):
        config = super().get_config()
        config.update({"lr_muon": self.lr_muon, "lr_1d": self.lr_1d,
                       "momentum": self.momentum_val, "weight_decay": self.weight_decay,
                       "ns_steps": self.ns_steps})
        return config


def create_optimizer(mode):
    """Create optimizer for the given benchmark configuration."""
    if mode == "hybrid_muon_adamw":
        return HybridMuonAdamW(
            lr_muon=0.02, lr_adam=5e-4, muon_momentum=0.95,
            beta1=0.9, beta2=0.95, weight_decay=0.01, ns_steps=5
        )
    elif mode == "adamw":
        return tf.keras.optimizers.AdamW(
            learning_rate=5e-4, beta_1=0.9, beta_2=0.95,
            weight_decay=0.01, epsilon=1e-8, clipnorm=1.0
        )
    elif mode == "muon":
        return PureMuon(
            lr_muon=0.02, lr_1d=5e-4, momentum=0.95,
            weight_decay=0.01, ns_steps=5
        )
    else:
        raise ValueError(f"Unknown optimizer: {mode}")


In [ ]:
# ==============================================================================
# CELL 04 — SEQUENTIAL 50M-TOKEN BENCHMARK: HYBRID → ADAMW → MUON
# ==============================================================================

def build_alpha_model():
    """Build Alpha 450M inside strategy.scope(). Returns model."""
    inp = layers.Input(shape=(SEQUENCE_LENGTH,), dtype=tf.int32, name="input_ids")
    emb = layers.Embedding(VOCAB_SIZE, HIDDEN_SIZE, name="tok_embeddings")
    x = emb(inp)
    for i in range(NUM_LAYERS):
        x = TransformerBlock(name=f"block_{i:02d}")(x)
    x = KerasRMSNorm(HIDDEN_SIZE, name="final_norm")(x)
    # Tied float32 logits: avoids bfloat16/float32 matmul dtype mismatch
    # (hidden state is bfloat16 under mixed precision, embeddings are float32).
    def _tied_logits(h):
        w = emb.weights[0] if emb.weights else emb.embeddings
        return tf.matmul(tf.cast(h, tf.float32), tf.cast(w, tf.float32), transpose_b=True)
    logits = layers.Lambda(_tied_logits, name="logits")(x)
    return models.Model(inputs=inp, outputs=logits, name="Alpha_450M")


def bench_dataset_fn(input_context):
    """Per-replica random dataset via distribute_datasets_from_function."""
    per_replica = input_context.get_per_replica_batch_size(GLOBAL_BATCH_SIZE)
    ds = tf.data.Dataset.range(1).repeat()
    ds = ds.map(
        lambda _: (
            tf.random.uniform((SEQUENCE_LENGTH,), 0, VOCAB_SIZE, dtype=tf.int32),
            tf.random.uniform((SEQUENCE_LENGTH,), 0, VOCAB_SIZE, dtype=tf.int32),
        ),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    ds = ds.batch(per_replica, drop_remainder=True)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


# ── Sequential Benchmark Run ──
CONFIGS = ["hybrid_muon_adamw", "adamw", "muon"]
results = []
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
param_count = None

for config_name in CONFIGS:
    print(f"\n{'=' * 64}")
    print(f"  🔧 Benchmarking: [{config_name.upper()}]")
    print(f"{'=' * 64}")

    # Clean slate between runs
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.mixed_precision.set_global_policy("mixed_bfloat16")

    # Build model + optimizer inside TPUStrategy scope
    with strategy.scope():
        model = build_alpha_model()
        opt = create_optimizer(config_name)
        model.compile(optimizer=opt, loss=loss_fn, steps_per_execution=STEPS_PER_EXEC)

    if param_count is None:
        param_count = model.count_params()
        print(f"  Parameters: {param_count:,}")

    bench_ds = strategy.distribute_datasets_from_function(bench_dataset_fn)

    # Warmup (triggers XLA compilation — NOT timed)
    print(f"  ⏳ Warmup ({WARMUP_STEPS} steps)...")
    model.fit(bench_ds, steps_per_epoch=WARMUP_STEPS, epochs=1, initial_epoch=0, verbose=0)

    # Timed benchmark
    print(f"  ⚡ Running {BENCH_STEPS}-step benchmark (~{BENCH_STEPS * TOKENS_PER_STEP:,} tokens)...")
    start_t = time.perf_counter()
    history = model.fit(bench_ds, steps_per_epoch=BENCH_STEPS, epochs=1, initial_epoch=0, verbose=1)
    elapsed = time.perf_counter() - start_t

    tokens_done = BENCH_STEPS * TOKENS_PER_STEP
    tok_s = tokens_done / elapsed
    final_loss = history.history["loss"][-1]
    hours_8_5b = (8_500_000_000 / tok_s) / 3600.0
    sessions = math.ceil(hours_8_5b / 8.4)

    results.append({
        "name": config_name.upper(),
        "tokens": tokens_done,
        "time_s": elapsed,
        "tok_s": tok_s,
        "loss": final_loss,
        "hours_8_5b": hours_8_5b,
        "sessions": sessions,
    })

    print(f"  ✅ {config_name.upper()} done: {tok_s:,.0f} tok/s | loss={final_loss:.4f}")
    del model, opt


# ── Combined Scorecard ──
print("\n\n" + "═" * 76)
print("  🏁 ALPHA 450M — 50M TOKEN BENCHMARK SCORECARD (TPU v5e-8)")
print("═" * 76)
header = f"  {'Optimizer':<22} {'Tokens':>12} {'Time':>8} {'tok/s':>12} {'Loss':>8} {'8.5B hrs':>9} {'Sessions':>9}"
print(header)
print("─" * 76)
for r in results:
    print(f"  {r['name']:<22} {r['tokens']:>12,} {r['time_s']:>7.1f}s {r['tok_s']:>12,.0f} {r['loss']:>8.4f} {r['hours_8_5b']:>8.1f}h {r['sessions']:>8}")
print("═" * 76)
print(f"\n  📊 Model: Alpha 450M ({param_count:,} params) | Per-step: {TOKENS_PER_STEP:,} tokens")
print(f"  💻 Hardware: TPU v5e-8 ({strategy.num_replicas_in_sync} cores) | Policy: mixed_bfloat16")
print(f"  ⚙️  XLA: steps_per_execution={STEPS_PER_EXEC} | Warmup: {WARMUP_STEPS} steps excluded")

# ── Save Scorecard to Disk for Persistence ──
# Works both on Kaggle (/kaggle/working) and locally (./)
OUTPUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
os.makedirs(OUTPUT_DIR, exist_ok=True)
scorecard_data = {
    "model": "Alpha 450M",
    "params": param_count,
    "hardware": f"TPU v5e-8 ({strategy.num_replicas_in_sync} cores)",
    "tokens_per_step": TOKENS_PER_STEP,
    "results": results
}
with open(os.path.join(OUTPUT_DIR, "scorecard.json"), "w") as f:
    json.dump(scorecard_data, f, indent=2)

scorecard_lines = [
    "=" * 76,
    "  ALPHA 450M — 50M TOKEN BENCHMARK SCORECARD (TPU v5e-8)",
    "=" * 76,
    header,
    "-" * 76,
]
for r in results:
    scorecard_lines.append(f"  {r['name']:<22} {r['tokens']:>12,} {r['time_s']:>7.1f}s {r['tok_s']:>12,.0f} {r['loss']:>8.4f} {r['hours_8_5b']:>8.1f}h {r['sessions']:>8}")
scorecard_lines.append("=" * 76)
with open(os.path.join(OUTPUT_DIR, "scorecard.txt"), "w") as f:
    f.write("\n".join(scorecard_lines) + "\n")
print(f"\n💾 Saved scorecard to {OUTPUT_DIR}/scorecard.json and scorecard.txt")
